In [16]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline

from scikeras.wrappers import KerasClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [17]:
df = pd.read_csv('Data\Churn_Modelling.csv')
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\singh\AppData\Local\Temp\ipykernel_22364\2071529278.py:1: SyntaxWarning: invalid escape sequence '\C'
  df = pd.read_csv('Data\Churn_Modelling.csv')


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [18]:
df = df.drop(columns= ['RowNumber', 'CustomerId', 'Surname'])

lb_gender = LabelEncoder()
df['Gender'] = lb_gender.fit_transform(df['Gender'])

onehot_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_geo.fit_transform(df[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns= onehot_geo.get_feature_names_out())

df = pd.concat([df.drop(columns=['Geography'], axis=1), geo_encoded_df], axis=1)

X = df.drop(columns= ['Exited'])
Y = df['Exited']

In [19]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [20]:
import pickle

with open("HyperParameters Models\lb_gender.pkl", 'wb') as file:
    pickle.dump(lb_gender, file)

with open("HyperParameters Models\onehot_geo.pkl", 'wb') as file:
    pickle.dump(onehot_geo, file)

with open("HyperParameters Models\scaler.pkl", 'wb') as file:
    pickle.dump(scaler, file)

<>:3: SyntaxWarning: invalid escape sequence '\l'
<>:6: SyntaxWarning: invalid escape sequence '\o'
<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\l'
<>:6: SyntaxWarning: invalid escape sequence '\o'
<>:9: SyntaxWarning: invalid escape sequence '\s'
C:\Users\singh\AppData\Local\Temp\ipykernel_22364\3150953108.py:3: SyntaxWarning: invalid escape sequence '\l'
  with open("HyperParameters Models\lb_gender.pkl", 'wb') as file:
C:\Users\singh\AppData\Local\Temp\ipykernel_22364\3150953108.py:6: SyntaxWarning: invalid escape sequence '\o'
  with open("HyperParameters Models\onehot_geo.pkl", 'wb') as file:
C:\Users\singh\AppData\Local\Temp\ipykernel_22364\3150953108.py:9: SyntaxWarning: invalid escape sequence '\s'
  with open("HyperParameters Models\scaler.pkl", 'wb') as file:


In [27]:
## Define a function to create a model and trying different parameters

def create_model(neurons=32, layers=1):
    model=Sequential()
    model.add(Dense(units=neurons, activation='relu', input_shape=(X_train.shape[1],)))

    for _ in range(layers - 1):
        model.add(Dense(units=neurons, activation='relu'))
    
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [28]:
## Create a Keras Classifier
model = KerasClassifier(model=create_model, verbose=0)

In [ ]:
## Define the grid search parameters
params = {
    'model__neurons': [16, 32, 64, 128],
    'model__layers': [1, 2, 3],
    'epochs': [50, 100],
    'batch_size': [10]
}

grid = GridSearchCV(estimator=model, param_grid=params, n_jobs=-1, cv=3)
grid_result = grid.fit(X_train, Y_train)

print("Best : %f using %s" % (grid_result.best_score_, grid_result.best_params_))